# BioRAG-X — 03 Chunking Baselines

## Fixed-size vs Recursive / Structure-Preserving Chunking

Notebook 02 created the canonical BioRAG-X knowledge layer.

This notebook starts the **chunking research track**. We deliberately begin with simple baselines so later semantic, biomedical-aware, proposition, parent-child, late, LLM-guided, agentic, and adaptive chunking strategies have a trustworthy reference.

### Core principle
**Chunking is a retrieval decision, not merely preprocessing.**

We will measure:
- representation quality
- content preservation
- chunk count / expansion
- boundary quality
- redundancy
- downstream retrieval readiness

We will NOT yet change embedding models, ANN, reranking, LLM generation, or retrieval policy.


## Learning goals

Understand from first principles:

- what chunking does
- why chunk size matters
- why overlap exists
- why fixed-size chunking remains a useful baseline
- why recursive chunking preserves structure better
- how bad boundaries can fragment evidence
- how to compare chunkers fairly

### Research hypotheses
1. Recursive chunking should create fewer arbitrary sentence boundaries than fixed chunking.
2. Moderate overlap should reduce boundary-related evidence loss.
3. Smaller chunks should improve localization but may lose context.
4. Larger chunks should improve context completeness but increase noise/cost.

These are hypotheses, not conclusions.


In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
import hashlib
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
CANONICAL_DIR = Path("data/canonical")
CHUNK_DIR = Path("data/chunks")
ARTIFACT_DIR = Path("artifacts/03_chunking_baselines")

CHUNK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = CANONICAL_DIR / "passages.parquet"
QUESTIONS_PATH = CANONICAL_DIR / "questions.parquet"
GOLD_REL_PATH = CANONICAL_DIR / "gold_relationships.parquet"

print("Canonical source:", PASSAGES_PATH)


Canonical source: data\canonical\passages.parquet


## 1. Load the canonical knowledge layer

In [2]:
if not PASSAGES_PATH.exists():
    raise FileNotFoundError("Notebook 02 must be run first: data/canonical/passages.parquet")

passages = pd.read_parquet(PASSAGES_PATH)
questions = pd.read_parquet(QUESTIONS_PATH) if QUESTIONS_PATH.exists() else None
gold_relationships = pd.read_parquet(GOLD_REL_PATH) if GOLD_REL_PATH.exists() else None

required = {"canonical_passage_id", "retrieval_text", "normalized_text"}
missing = required - set(passages.columns)
if missing:
    raise ValueError(f"Missing canonical fields: {sorted(missing)}")

passages["retrieval_text"] = passages["retrieval_text"].fillna("").astype(str)

print("Passages:", len(passages))
print("Questions:", None if questions is None else len(questions))
print("Gold relationships:", None if gold_relationships is None else len(gold_relationships))
display(passages.head(3))


Passages: 40221
Questions: 4719
Gold relationships: 42608


,passage,id,source_dataset,source_config,source_split,source_row_id,source_passage_id,is_empty,raw_text,is_usable,...,acronym_density,hyphenated_density,digit_token_density,negation_cue_count,negation_density,reference_marker_count,greek_char_count,avg_sentence_length_words,retrieval_text,metadata_json
0,New data on viruses isolated from patients wit...,9797,rag-datasets/rag-mini-bioasq,text-corpus,passages,0,9797,False,New data on viruses isolated from patients wit...,True,...,0.000000,0.020408,0.000000,0.0,0.000000,0.0,0.0,12.250000,New data on viruses isolated from patients wit...,"{""exact_duplicate_group_id"": ""DG-00000000"", ""i..."
1,We describe an improved method for detecting d...,11906,rag-datasets/rag-mini-bioasq,text-corpus,passages,1,11906,False,We describe an improved method for detecting d...,True,...,0.016129,0.064516,0.032258,0.0,0.000000,0.0,0.0,20.666667,We describe an improved method for detecting d...,"{""exact_duplicate_group_id"": ""DG-00000001"", ""i..."
2,We have studied the effects of curare on respo...,16083,rag-datasets/rag-mini-bioasq,text-corpus,passages,2,16083,False,We have studied the effects of curare on respo...,True,...,0.009901,0.019802,0.019802,2.0,0.222222,0.0,0.0,22.444444,We have studied the effects of curare on respo...,"{""exact_duplicate_group_id"": ""DG-00000002"", ""i..."


## 2. Inspect the source passage distribution

In [3]:
TOKEN_RE = re.compile(r"\S+")

def approx_tokens(text):
    return TOKEN_RE.findall(text or "")

def sentence_count(text):
    text = (text or "").strip()
    if not text:
        return 0
    return len(re.findall(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)) + 1

passages["approx_tokens"] = passages["retrieval_text"].map(lambda x: len(approx_tokens(x)))
passages["char_count"] = passages["retrieval_text"].str.len()
passages["sentence_count"] = passages["retrieval_text"].map(sentence_count)

display(passages[["approx_tokens", "char_count", "sentence_count"]].describe(
    percentiles=[.5, .9, .95, .99]
))


,approx_tokens,char_count,sentence_count
count,40221.000000,40221.000000,40221.000000
mean,146.222322,1019.342955,6.330524
std,126.904175,881.046650,5.283556
min,0.000000,0.000000,0.000000
50%,163.000000,1148.000000,7.000000
90%,275.000000,1915.000000,13.000000
95%,311.000000,2172.000000,14.000000
99%,431.000000,2949.000000,19.000000
max,4215.000000,33088.000000,76.000000


## 3. Fixed-size chunking

The baseline is:

`document → N tokens with overlap O`

Strengths:
- deterministic
- fast
- easy to reproduce
- easy to scale

Risks:
- sentence fragmentation
- entity/relation fragmentation
- overlap-driven redundancy

This is a **baseline**, not the expected winner.


In [4]:
@dataclass
class ChunkRecord:
    chunk_id: str
    parent_passage_id: str
    strategy: str
    chunk_index: int
    text: str
    approximate_tokens: int
    char_count: int
    start_token: int
    end_token: int
    overlap_tokens: int
    metadata_json: str

def stable_chunk_id(parent_id, strategy, index, text):
    payload = f"{parent_id}|{strategy}|{index}|{text}".encode("utf-8")
    return "CHK-" + hashlib.sha256(payload).hexdigest()[:20]

def fixed_chunks(parent_id, text, chunk_size=256, overlap=32):
    tokens = approx_tokens(text)
    if not tokens:
        return []
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    step = chunk_size - overlap
    records = []
    start = 0
    idx = 0

    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = " ".join(chunk_tokens)

        records.append(ChunkRecord(
            chunk_id=stable_chunk_id(parent_id, "fixed", idx, chunk_text),
            parent_passage_id=parent_id,
            strategy="fixed",
            chunk_index=idx,
            text=chunk_text,
            approximate_tokens=len(chunk_tokens),
            char_count=len(chunk_text),
            start_token=start,
            end_token=end,
            overlap_tokens=0 if idx == 0 else overlap,
            metadata_json=json.dumps(
                {"chunk_size": chunk_size, "overlap": overlap},
                sort_keys=True
            ),
        ))

        idx += 1
        if end == len(tokens):
            break
        start += step

    return records

demo = fixed_chunks(
    "BIOP-DEMO",
    "Sentence one. Sentence two contains an important biomedical relationship. Sentence three adds context.",
    chunk_size=6,
    overlap=2
)
display(pd.DataFrame(asdict(x) for x in demo))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-32cefbc75b425bfda1f4,BIOP-DEMO,fixed,0,Sentence one. Sentence two contains an,6,38,0,6,0,"{""chunk_size"": 6, ""overlap"": 2}"
1,CHK-c83bc7bfd656a767d1ff,BIOP-DEMO,fixed,1,contains an important biomedical relationship....,6,55,4,10,2,"{""chunk_size"": 6, ""overlap"": 2}"
2,CHK-9770232b599726c16f21,BIOP-DEMO,fixed,2,relationship. Sentence three adds context.,5,42,8,13,2,"{""chunk_size"": 6, ""overlap"": 2}"


## 4. Fixed-chunk boundary diagnostics

We flag simple signs of awkward splitting:

- likely mid-sentence start
- likely mid-sentence end
- unbalanced parentheses

These are diagnostic heuristics. They do not replace actual retrieval evaluation.


In [5]:
def boundary_flags(text):
    text = (text or "").strip()
    if not text:
        return {
            "starts_mid_sentence": False,
            "ends_mid_sentence": False,
            "unbalanced_parentheses": False,
        }

    # Heuristic: chunk starts with a continuation-like token.
    starts_mid = bool(re.match(r'^[a-z]', text))
    ends_mid = not bool(re.search(r'[.!?\)\]"\']$', text))

    return {
        "starts_mid_sentence": starts_mid,
        "ends_mid_sentence": ends_mid,
        "unbalanced_parentheses": text.count("(") != text.count(")"),
    }

display(pd.DataFrame([boundary_flags(c.text) for c in demo]))


,starts_mid_sentence,ends_mid_sentence,unbalanced_parentheses
0,False,True,False
1,True,True,False
2,True,False,False


## 5. Recursive / structure-preserving chunking

Recursive chunking tries:

1. paragraph/block
2. sentence
3. token

A chunk is split only when the current structural unit exceeds the maximum size.

This preserves document structure without requiring embeddings or an LLM.


In [6]:
def split_sentences(text):
    text = (text or "").strip()
    if not text:
        return []
    return [s.strip() for s in re.split(
        r"(?<=[.!?])\s+(?=[A-Z0-9])", text
    ) if s.strip()]

def recursive_chunks(
    parent_id,
    text,
    max_tokens=256,
    min_tokens=32,
):
    text = (text or "").strip()
    if not text:
        return []

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    units = []

    for paragraph in paragraphs:
        p_tokens = approx_tokens(paragraph)
        if len(p_tokens) <= max_tokens:
            units.append(paragraph)
            continue

        sentences = split_sentences(paragraph)

        if len(sentences) <= 1:
            for i in range(0, len(p_tokens), max_tokens):
                units.append(" ".join(p_tokens[i:i + max_tokens]))
            continue

        current = []
        current_len = 0

        for sentence in sentences:
            s_tokens = approx_tokens(sentence)

            if len(s_tokens) > max_tokens:
                if current:
                    units.append(" ".join(current))
                    current, current_len = [], 0
                for i in range(0, len(s_tokens), max_tokens):
                    units.append(" ".join(s_tokens[i:i + max_tokens]))
                continue

            if current and current_len + len(s_tokens) > max_tokens:
                units.append(" ".join(current))
                current = [sentence]
                current_len = len(s_tokens)
            else:
                current.append(sentence)
                current_len += len(s_tokens)

        if current:
            units.append(" ".join(current))

    # Merge tiny adjacent units where possible.
    merged = []
    for unit in units:
        if merged and len(approx_tokens(unit)) < min_tokens:
            candidate = f"{merged[-1]} {unit}".strip()
            if len(approx_tokens(candidate)) <= max_tokens:
                merged[-1] = candidate
            else:
                merged.append(unit)
        else:
            merged.append(unit)

    records = []
    for idx, unit in enumerate(merged):
        chunk_text = " ".join(approx_tokens(unit))
        records.append(ChunkRecord(
            chunk_id=stable_chunk_id(parent_id, "recursive", idx, chunk_text),
            parent_passage_id=parent_id,
            strategy="recursive",
            chunk_index=idx,
            text=chunk_text,
            approximate_tokens=len(approx_tokens(chunk_text)),
            char_count=len(chunk_text),
            start_token=-1,
            end_token=-1,
            overlap_tokens=0,
            metadata_json=json.dumps({
                "max_tokens": max_tokens,
                "min_tokens": min_tokens,
                "structural_priority": ["paragraph", "sentence", "token"]
            }, sort_keys=True),
        ))

    return records

recursive_demo = recursive_chunks(
    "BIOP-DEMO",
    "Paragraph one contains a short idea. It has two sentences.\n\n"
    "Paragraph two explains an important biomedical mechanism. It has several "
    "related statements. Another sentence continues the explanation.",
    max_tokens=12,
    min_tokens=4
)
display(pd.DataFrame(asdict(x) for x in recursive_demo))


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-3f75aef5d3bf8dae7626,BIOP-DEMO,recursive,0,Paragraph one contains a short idea. It has tw...,10,58,-1,-1,0,"{""max_tokens"": 12, ""min_tokens"": 4, ""structura..."
1,CHK-57548b853ac5b1c02aa3,BIOP-DEMO,recursive,1,Paragraph two explains an important biomedical...,12,92,-1,-1,0,"{""max_tokens"": 12, ""min_tokens"": 4, ""structura..."
2,CHK-c208b179da298bf77a80,BIOP-DEMO,recursive,2,Another sentence continues the explanation.,5,43,-1,-1,0,"{""max_tokens"": 12, ""min_tokens"": 4, ""structura..."


## 6. Apply both baselines to the full corpus

In [7]:
def chunk_corpus_fixed(df, chunk_size, overlap):
    out = []
    for row in df[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
        out.extend(fixed_chunks(row.canonical_passage_id, row.retrieval_text,
                                 chunk_size=chunk_size, overlap=overlap))
    return pd.DataFrame(asdict(x) for x in out)

def chunk_corpus_recursive(df, max_tokens, min_tokens):
    out = []
    for row in df[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
        out.extend(recursive_chunks(row.canonical_passage_id, row.retrieval_text,
                                    max_tokens=max_tokens, min_tokens=min_tokens))
    return pd.DataFrame(asdict(x) for x in out)

fixed_256 = chunk_corpus_fixed(passages, 256, 32)
recursive_256 = chunk_corpus_recursive(passages, 256, 32)

print("Fixed chunks:", len(fixed_256))
print("Recursive chunks:", len(recursive_256))

display(fixed_256.head(3))
display(recursive_256.head(3))


Fixed chunks: 34237
Recursive chunks: 34205


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-2d4a16b240818e1214c3,BIOP-9797,fixed,0,New data on viruses isolated from patients wit...,49,355,0,49,0,"{""chunk_size"": 256, ""overlap"": 32}"
1,CHK-b90f4ff740d3caf3f7a2,BIOP-11906,fixed,0,We describe an improved method for detecting d...,62,445,0,62,0,"{""chunk_size"": 256, ""overlap"": 32}"
2,CHK-572b99ef43c7f98bdfb3,BIOP-16083,fixed,0,We have studied the effects of curare on respo...,202,1390,0,202,0,"{""chunk_size"": 256, ""overlap"": 32}"


,chunk_id,parent_passage_id,strategy,chunk_index,text,approximate_tokens,char_count,start_token,end_token,overlap_tokens,metadata_json
0,CHK-832f39e7050eb6418539,BIOP-9797,recursive,0,New data on viruses isolated from patients wit...,49,355,-1,-1,0,"{""max_tokens"": 256, ""min_tokens"": 32, ""structu..."
1,CHK-5d6389182f7d541f0538,BIOP-11906,recursive,0,We describe an improved method for detecting d...,62,445,-1,-1,0,"{""max_tokens"": 256, ""min_tokens"": 32, ""structu..."
2,CHK-192be9c368899b69321f,BIOP-16083,recursive,0,We have studied the effects of curare on respo...,202,1390,-1,-1,0,"{""max_tokens"": 256, ""min_tokens"": 32, ""structu..."


## 7. Chunk economics

In [8]:
def duplicate_text_ratio(chunks):
    normalized = chunks["text"].map(
        lambda x: re.sub(r"\s+", " ", x.lower()).strip()
    )
    return normalized.duplicated().mean()

def summarize(chunks, strategy):
    return {
        "strategy": strategy,
        "chunks": len(chunks),
        "source_passages": len(passages),
        "chunks_per_passage": len(chunks) / max(1, len(passages)),
        "median_tokens": chunks["approximate_tokens"].median(),
        "p95_tokens": chunks["approximate_tokens"].quantile(.95),
        "duplicate_text_ratio": duplicate_text_ratio(chunks),
        "empty_chunks": int((chunks["approximate_tokens"] == 0).sum()),
    }

economics = pd.DataFrame([
    summarize(fixed_256, "fixed_256_32"),
    summarize(recursive_256, "recursive_256"),
])

display(economics)


,strategy,chunks,source_passages,chunks_per_passage,median_tokens,p95_tokens,duplicate_text_ratio,empty_chunks
0,fixed_256_32,34237,40221,0.851222,192.0,256.0,0.000876,0
1,recursive_256,34205,40221,0.850426,191.0,253.0,0.000965,0


## 8. Boundary diagnostics on both baselines

In [9]:
def boundary_summary(chunks):
    flags = chunks["text"].map(boundary_flags).apply(pd.Series)
    return {
        "start_mid_sentence_rate": flags["starts_mid_sentence"].mean(),
        "end_mid_sentence_rate": flags["ends_mid_sentence"].mean(),
        "unbalanced_parentheses_rate": flags["unbalanced_parentheses"].mean(),
    }

boundary = pd.DataFrame({
    "fixed": boundary_summary(fixed_256),
    "recursive": boundary_summary(recursive_256)
}).T

display(boundary)


,start_mid_sentence_rate,end_mid_sentence_rate,unbalanced_parentheses_rate
fixed,0.138155,0.175804,0.028040
recursive,0.003070,0.010466,0.011899


## 9. Content preservation / reconstructability

We preserve `parent_passage_id`, so later gold-evidence evaluation can operate at either:

- **chunk level**: exact retrieved chunk
- **parent level**: any chunk from the gold passage

For this notebook we perform a token-content coverage test. The target is **no accidental content loss**, not byte-identical reconstruction.


In [10]:
def parent_coverage(passages_df, chunks_df):
    chunk_map = chunks_df.groupby("parent_passage_id")["text"].apply(lambda s: " ".join(s))
    rows = []

    for row in passages_df[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
        source_tokens = approx_tokens(row.retrieval_text.lower())
        combined = chunk_map.get(row.canonical_passage_id, "")
        chunk_tokens = approx_tokens(combined.lower())

        src = Counter(source_tokens)
        got = Counter(chunk_tokens)

        covered = sum(min(n, got.get(tok, 0)) for tok, n in src.items())

        rows.append({
            "parent_passage_id": row.canonical_passage_id,
            "coverage": covered / max(1, len(source_tokens)),
            "source_tokens": len(source_tokens),
            "chunk_tokens": len(chunk_tokens),
        })

    return pd.DataFrame(rows)

fixed_cov = parent_coverage(passages, fixed_256)
recursive_cov = parent_coverage(passages, recursive_256)

display(pd.DataFrame({
    "fixed": fixed_cov["coverage"].describe(),
    "recursive": recursive_cov["coverage"].describe(),
}).T)


,count,mean,std,min,25%,50%,75%,max
fixed,40221.0,0.696179,0.459912,0.0,0.0,1.0,1.0,1.0
recursive,40221.0,0.696179,0.459912,0.0,0.0,1.0,1.0,1.0


## 10. Fixed-size chunk grid

We test a small controlled grid:

| Size | Overlap |
|---:|---:|
| 128 | 16 |
| 256 | 32 |
| 512 | 64 |
| 1024 | 128 |

This is an **economics/boundary experiment**, not yet the final retrieval benchmark.


In [11]:
grid_rows = []

for size, overlap in [(128, 16), (256, 32), (512, 64), (1024, 128)]:
    chunks = chunk_corpus_fixed(passages, size, overlap)
    b = boundary_summary(chunks)

    grid_rows.append({
        "chunk_size": size,
        "overlap": overlap,
        "chunks": len(chunks),
        "chunks_per_passage": len(chunks) / len(passages),
        "median_tokens": chunks["approximate_tokens"].median(),
        "p95_tokens": chunks["approximate_tokens"].quantile(.95),
        "duplicate_text_ratio": duplicate_text_ratio(chunks),
        "start_mid_sentence_rate": b["start_mid_sentence_rate"],
        "end_mid_sentence_rate": b["end_mid_sentence_rate"],
    })

size_grid = pd.DataFrame(grid_rows)
display(size_grid)


,chunk_size,overlap,chunks,chunks_per_passage,median_tokens,p95_tokens,duplicate_text_ratio,start_mid_sentence_rate,end_mid_sentence_rate
0,128,16,62938,1.564804,128.0,128.0,0.000620,0.446074,0.526645
1,256,32,34237,0.851222,192.0,256.0,0.000876,0.138155,0.175804
2,512,64,28284,0.703215,204.0,343.0,0.000955,0.007495,0.012374
3,1024,128,28047,0.697322,204.0,339.0,0.000963,0.003173,0.004314


## 11. Gold-evidence compatibility

In [12]:
if gold_relationships is not None:
    gold_ids = set(gold_relationships["canonical_passage_id"])
    fixed_parents = set(fixed_256["parent_passage_id"])
    recursive_parents = set(recursive_256["parent_passage_id"])

    compatibility = pd.DataFrame({
        "metric": [
            "unique gold passages",
            "gold passages represented in fixed",
            "gold passages represented in recursive",
        ],
        "value": [
            len(gold_ids),
            len(gold_ids & fixed_parents),
            len(gold_ids & recursive_parents),
        ],
    })
    compatibility["coverage_pct"] = compatibility["value"] / max(1, len(gold_ids)) * 100
    display(compatibility)
else:
    print("Gold relationships not found. Run Notebook 02 first.")


,metric,value,coverage_pct
0,unique gold passages,40221,100.000000
1,gold passages represented in fixed,28001,69.617861
2,gold passages represented in recursive,28001,69.617861


## 7b. Real-tokenizer sanity check

All sizing above uses **approximate whitespace tokens** (`\S+`). Downstream embedding models (Notebook 06) use subword tokenizers, so a whitespace-token budget can differ from the real model-token budget — especially for biomedical text (hyphenated entities, IDs, Greek letters). We quantify the inflation factor here so the baseline sizes are interpretable.

This uses `tiktoken` for a sanity check only. **No embedding model or LLM is loaded.**

In [13]:
# Real vs approximate token ratio (sanity check only; no model inference).
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def real_tokens(text):
        return len(_enc.encode(text or ""))
    _tok_backend = "tiktoken/cl100k_base"
except Exception as e:  # pragma: no cover - fallback keeps notebook runnable
    def real_tokens(text):
        return len(approx_tokens(text))
    _tok_backend = f"whitespace-fallback ({e})"

_sample = fixed_256.sample(min(2000, len(fixed_256)), random_state=SEED).copy()
_sample["real_tokens"] = _sample["text"].map(real_tokens)
_sample["ratio"] = _sample["real_tokens"] / _sample["approximate_tokens"].clip(lower=1)

tokenizer_sanity = pd.DataFrame({
    "backend": [_tok_backend],
    "sampled_chunks": [len(_sample)],
    "mean_whitespace_tokens": [_sample["approximate_tokens"].mean()],
    "mean_real_tokens": [_sample["real_tokens"].mean()],
    "median_ratio_real_over_ws": [_sample["ratio"].median()],
    "p95_ratio_real_over_ws": [_sample["ratio"].quantile(.95)],
})
print("Token backend:", _tok_backend)
print("Interpretation: a 256 whitespace-token chunk is ~",
      round(float(_sample['ratio'].median()) * 256), "real model tokens (median).")
display(tokenizer_sanity)


Token backend: tiktoken/cl100k_base
Interpretation: a 256 whitespace-token chunk is ~ 384 real model tokens (median).


,backend,sampled_chunks,mean_whitespace_tokens,mean_real_tokens,median_ratio_real_over_ws,p95_ratio_real_over_ws
0,tiktoken/cl100k_base,2000,176.916,272.668,1.5,1.924196


## 7c. Spec-aligned fixed baseline (384 tokens, ~12% overlap)

`discussionAndResearch.md` section 8.B specifies the fixed/recursive baseline operating range as **384-512 tokens with 10-15% overlap**. The primary `256/32` config above sits below that range, so we materialize the spec-aligned `384/46` baseline (46 ~= 12% of 384) for a like-for-like record. We keep `256/32` as the comparison anchor because most BioASQ passages are short abstracts (median ~163 whitespace tokens), so 384 rarely splits them.

In [14]:
# Spec-aligned baseline: 384 tokens / ~12% overlap (doc section 8.B).
fixed_384 = chunk_corpus_fixed(passages, 384, 46)
print("Fixed 384/46 chunks:", len(fixed_384))

spec_economics = pd.DataFrame([
    summarize(fixed_256, "fixed_256_32"),
    summarize(fixed_384, "fixed_384_46"),
    summarize(recursive_256, "recursive_256"),
])
display(spec_economics)


Fixed 384/46 chunks: 28848


,strategy,chunks,source_passages,chunks_per_passage,median_tokens,p95_tokens,duplicate_text_ratio,empty_chunks
0,fixed_256_32,34237,40221,0.851222,192.0,256.0,0.000876,0
1,fixed_384_46,28848,40221,0.717237,203.0,344.0,0.000936,0
2,recursive_256,34205,40221,0.850426,191.0,253.0,0.000965,0


## 8b. Biomedical-aware boundary heuristics

The simple `boundary_flags` in section 4 treats any trailing period as a clean sentence end, but biomedical text ends many non-final tokens with periods (`e.g.`, `i.v.`, `Fig.`, `spp.`, `vs.`) and uses square brackets `[]` for citations. We add a refined check that (a) ignores common abbreviation-final periods and (b) also tests bracket balance. This gives a less noisy read on boundary quality without changing the section-4 baseline numbers.

In [15]:
_ABBREV = {
    "e.g.", "i.e.", "i.v.", "i.p.", "i.m.", "vs.", "fig.", "figs.", "eq.",
    "no.", "spp.", "sp.", "cf.", "approx.", "et al.", "ref.", "refs.", "ca.",
}
_ABBREV_RE = re.compile(r"(?:" + "|".join(re.escape(a) for a in _ABBREV) + r")$", re.IGNORECASE)

def boundary_flags_bio(text):
    text = (text or "").strip()
    if not text:
        return {"starts_mid_sentence": False, "ends_mid_sentence": False,
                "unbalanced_parentheses": False, "unbalanced_brackets": False}
    starts_mid = bool(re.match(r"^[a-z]", text))
    ends_clean = bool(re.search(r"[.!?\)\]\"']$", text)) and not _ABBREV_RE.search(text)
    return {
        "starts_mid_sentence": starts_mid,
        "ends_mid_sentence": not ends_clean,
        "unbalanced_parentheses": text.count("(") != text.count(")"),
        "unbalanced_brackets": text.count("[") != text.count("]"),
    }

def boundary_summary_bio(chunks):
    flags = chunks["text"].map(boundary_flags_bio).apply(pd.Series)
    return {
        "start_mid_sentence_rate": flags["starts_mid_sentence"].mean(),
        "end_mid_sentence_rate": flags["ends_mid_sentence"].mean(),
        "unbalanced_parentheses_rate": flags["unbalanced_parentheses"].mean(),
        "unbalanced_brackets_rate": flags["unbalanced_brackets"].mean(),
    }

boundary_bio = pd.DataFrame({
    "fixed_256": boundary_summary_bio(fixed_256),
    "fixed_384": boundary_summary_bio(fixed_384),
    "recursive_256": boundary_summary_bio(recursive_256),
}).T
display(boundary_bio)


,start_mid_sentence_rate,end_mid_sentence_rate,unbalanced_parentheses_rate,unbalanced_brackets_rate
fixed_256,0.138155,0.178111,0.028040,0.003038
fixed_384,0.022012,0.033105,0.014698,0.002253
recursive_256,0.003070,0.013799,0.011899,0.002105


## 10b. Parent-child representation (baseline tier)

`discussionAndResearch.md` section 8.C lists parent-child as a baseline-tier representation: **search small children, return the parent passage to the generator**. This avoids the classic failure where the exact answer chunk is retrieved but its explanatory context is lost. We materialize child chunks (128/16) each carrying a `parent_passage_id` link back to the full passage. Parent text is the canonical passage itself, so we store only the child->parent index here (no text duplication).

In [16]:
# Parent-child: children = small fixed chunks (128/16); parent = full canonical passage.
children_128 = chunk_corpus_fixed(passages, 128, 16).copy()
children_128["strategy"] = "parent_child_child"

children_per_parent = children_128.groupby("parent_passage_id").size()
parent_child_summary = pd.DataFrame([{
    "child_chunks": len(children_128),
    "parents_with_children": children_per_parent.size,
    "mean_children_per_parent": children_per_parent.mean(),
    "max_children_per_parent": int(children_per_parent.max()),
    "single_child_parents_pct": float((children_per_parent == 1).mean() * 100),
}])
display(parent_child_summary)


,child_chunks,parents_with_children,mean_children_per_parent,max_children_per_parent,single_child_parents_pct
0,62938,28001,2.247705,38,14.035213


## 11b. Cheap gold-evidence proxy (no retrieval, no embeddings)

The notebook defers real retrieval metrics to Notebook 07. But we can still get an early read on hypotheses 1-2 (fragmentation / boundary-driven evidence loss) with a cheap, deterministic proxy:

For each question that has a usable gold passage present in our chunk set, we compute the **answer content-token recall of the single best chunk** of that gold passage. A chunking that fragments evidence forces the answer content across multiple chunks, lowering the best-single-chunk recall. We compare fixed vs recursive. This is a proxy, not a retrieval benchmark.

In [17]:
# Answer content-token recall within the BEST single chunk of the gold passage.
_STOP = set("the a an of and or to in for with on by is are was were be as at from that this "
            "these those we our it its their his her can may using used based between within into "
            "which who whom whose than then also not no such via per over under more most less".split())
_WORD_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9\-]+")

def content_tokens(text):
    toks = [t.lower() for t in _WORD_RE.findall(text or "")]
    return [t for t in toks if t not in _STOP and len(t) > 2]

def best_single_chunk_recall(chunks_df):
    # Map parent -> list of chunk token sets (as Counters).
    by_parent = {}
    for pid, grp in chunks_df.groupby("parent_passage_id"):
        by_parent[pid] = [Counter(content_tokens(t)) for t in grp["text"]]

    usable = questions[questions["has_usable_gold_evidence"] == True]
    recalls = []
    for row in usable[["answer", "gold_canonical_passage_ids"]].itertuples(index=False):
        ans = Counter(content_tokens(row.answer))
        if not ans:
            continue
        total = sum(ans.values())
        best = 0.0
        gold_ids = row.gold_canonical_passage_ids
        gold_ids = list(gold_ids) if gold_ids is not None else []
        for gid in gold_ids:
            for ch in by_parent.get(gid, []):
                covered = sum(min(n, ch.get(tok, 0)) for tok, n in ans.items())
                r = covered / total
                if r > best:
                    best = r
                    if best >= 0.999:
                        break
            if best >= 0.999:
                break
        recalls.append(best)
    return pd.Series(recalls, dtype=float)

fixed_gold_recall = best_single_chunk_recall(fixed_256)
recursive_gold_recall = best_single_chunk_recall(recursive_256)

gold_proxy = pd.DataFrame({
    "fixed_256": fixed_gold_recall.describe(),
    "recursive_256": recursive_gold_recall.describe(),
}).T
print("Questions evaluated:", len(fixed_gold_recall))
print("Mean best-single-chunk answer recall -> fixed:",
      round(float(fixed_gold_recall.mean()), 4),
      "| recursive:", round(float(recursive_gold_recall.mean()), 4))
display(gold_proxy)


Questions evaluated: 4385
Mean best-single-chunk answer recall -> fixed: 0.6897 | recursive: 0.6881


,count,mean,std,min,25%,50%,75%,max
fixed_256,4385.0,0.689672,0.270259,0.0,0.5,0.714286,0.953125,1.0
recursive_256,4385.0,0.688135,0.270396,0.0,0.5,0.714286,0.952381,1.0


## 12. Persist baseline artifacts

In [18]:
fixed_path = CHUNK_DIR / "fixed_256_32.parquet"
recursive_path = CHUNK_DIR / "recursive_256.parquet"

fixed_256.to_parquet(fixed_path, index=False)
recursive_256.to_parquet(recursive_path, index=False)
size_grid.to_parquet(ARTIFACT_DIR / "fixed_size_grid.parquet", index=False)
economics.to_csv(ARTIFACT_DIR / "chunk_economics.csv", index=False)
boundary.to_csv(ARTIFACT_DIR / "boundary_quality.csv")

manifest = {
    "notebook": "03_chunking_baselines",
    "seed": SEED,
    "token_unit": "approximate_whitespace_tokens",
    "baselines": {
        "fixed_256_32": str(fixed_path),
        "recursive_256": str(recursive_path),
    },
    "research_status": {
        "retrieval_evaluation_completed": False,
        "embedding_evaluation_completed": False,
        "winner_selected": False,
    }
}

manifest_path = ARTIFACT_DIR / "chunking_baseline_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved baseline chunks and artifacts.")

# --- NB03 baseline-improvement artifacts (added) ---
fixed_384.to_parquet(CHUNK_DIR / "fixed_384_46.parquet", index=False)
children_128.to_parquet(CHUNK_DIR / "parent_child_children_128_16.parquet", index=False)
tokenizer_sanity.to_csv(ARTIFACT_DIR / "tokenizer_sanity.csv", index=False)
spec_economics.to_csv(ARTIFACT_DIR / "chunk_economics_spec_aligned.csv", index=False)
boundary_bio.to_csv(ARTIFACT_DIR / "boundary_quality_biomedical.csv")
parent_child_summary.to_csv(ARTIFACT_DIR / "parent_child_summary.csv", index=False)
gold_proxy.to_csv(ARTIFACT_DIR / "gold_evidence_proxy.csv")

manifest["baselines"]["fixed_384_46"] = str(CHUNK_DIR / "fixed_384_46.parquet")
manifest["baselines"]["parent_child_children_128_16"] = str(CHUNK_DIR / "parent_child_children_128_16.parquet")
manifest["token_sanity_backend"] = _tok_backend
manifest["proxy_metrics"] = {
    "gold_answer_recall_fixed_256": float(fixed_gold_recall.mean()),
    "gold_answer_recall_recursive_256": float(recursive_gold_recall.mean()),
    "note": "best-single-chunk answer content-token recall; proxy only, not retrieval eval",
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


Saved baseline chunks and artifacts.


822

# 13. What we learned

### Fixed-size
**Pros:** simple, deterministic, scalable, reproducible.

**Cons:** arbitrary boundaries and overlap redundancy.

### Recursive
**Pros:** respects paragraph/sentence structure, better interpretability.

**Cons:** still heuristic; long sentences and unusual biomedical structure can remain problematic.

### Important conclusion

**We do not choose the winner here.**

The next stage must test whether these representation differences actually improve:

- retrieval Recall@K
- multi-passage evidence recall
- MRR / nDCG
- evidence precision
- answer correctness
- faithfulness
- latency
- index size

That controlled downstream experiment is what turns "chunking opinions" into evidence.


# 14. Handoff to Notebook 04

Notebook 04 will introduce the advanced representation family:

- Semantic chunking
- Biomedical-aware / entity-aware chunking
- Proposition / atomic evidence
- Parent-child hierarchical representation
- Late chunking

Notebook 05 will then cover:

- LLM-guided chunking
- Agentic chunking
- Adaptive/document-aware chunking router

**All future chunkers must preserve `parent_passage_id`** so we can compare them against the same BioASQ gold evidence.
